# Environment Setup

Clone the LLaMa-Factory repository (default loss function).

In [ ]:
%cd /content/
%rm -rf LLaMA* sample_data
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%mkdir -p LLaMA-Factory/data/ADPr
%cp *.csv LLaMA-Factory/data/ADPr
%pwd
# %cp adpr-fine-tuning-config.yaml LLaMA-Factory/config/user_config.yaml
%cd LLaMA-Factory
%ls

/content
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 362, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (277/277), done.
remote: Total 362 (delta 76), reused 293 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (362/362), 9.95 MiB | 27.84 MiB/s, done.
Resolving deltas: 100% (76/76), done.
/content/LLaMA-Factory
assets/       evaluation/  MANIFEST.in     requirements.txt  tests/
CITATION.cff  examples/    pyproject.toml  scripts/
data/         LICENSE      README.md       setup.py
docker/       Makefile     README_zh.md    src/


Install dependencies.

In [ ]:
!pip uninstall -y jax
!pip install -e .[torch,bitsandbytes,liger-kernel]
!pip install llamafactory
!pip show llamafactory
!pip install scikit-learn
!pip install peft
!pip install accelerate
!git lfs install

Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.4.dev0-0.editable-py3-none-any.whl size=27716 sha256=c400f34a860b4c1da95f609552c3b3a37bfc473bb4dcf691837640e1ece85d66
  Stored in directory: /tmp/pip-ephem-wheel-cache-nzhhh86_/wheels/bd/34/05/1e3cb4b8f20c20631b411dc5157b4b150850c03496fa96c2c4
Successfully built llamafactory
  Attempting uninstall: llamafactory
    Found existing installation: llamafactory 0.9.4.dev0
    Uninstalling llamafactory-0.9.4.dev0:
      Successfully uninstalled llamafactory-0.9.4.dev0
Name: llamafactory
Version: 0.9.4.dev0
Summary: Unified Efficient Fine-Tuning of 100+ LLMs
Home-page: https://github.com/hiyouga/LLaMA-Factory
Author: hiyouga
Aut

Import libraries.

In [ ]:
import numpy as np
import pandas as pd
import torch
import json
import os
import transformers
import huggingface_hub
import accelerate
import tokenizers
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM
from torch.quantization import quantize_dynamic
from sklearn.model_selection import train_test_split
from huggingface_hub import notebook_login, HfApi, Repository
from google.colab import userdata

print("torch version:", torch.__version__)
print("transformers version:", transformers.__version__)
print("huggingface_hub version:", huggingface_hub.__version__)
print("accelerate version:", accelerate.__version__)
print("tokenizers version:", tokenizers.__version__)

torch version: 2.6.0+cu124
transformers version: 4.52.4
huggingface_hub version: 0.33.4
accelerate version: 1.7.0
tokenizers version: 0.21.1


Assert GPU availability.

In [ ]:
try:
  assert torch.cuda.is_available() is True
  print("GPU is available!")
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory")

GPU is available!


# Dataset Preparation
We load the tabular datasets and convert them into a JSON instruction dataset format that will be used to fine tune the model.

In [ ]:
def prep_dataset(df, instruction):
    # # Apply filter to only include positive sites
    df = df[df['has_ptm'] == 1].copy()  # Explicitly create a copy

    # Create instruction column
    df.loc[:, 'instruction'] = instruction

    # Wrap the sequence in ProLLaMA format prefix and suffix
    df.loc[:, 'seq_sequence'] = "Seq=<" + df['seq_sequence'] + ">"

    # Wrap the PTM sites and strip whitespace
    df['ADPr_binding_sites'] = df['ADPr_binding_sites'].fillna('')
    df.loc[:, 'ADPr_binding_sites'] = "Sites=<" + df['ADPr_binding_sites'] + ">"
    df.loc[:, 'ADPr_binding_sites'] = df.loc[:, 'ADPr_binding_sites'].str.replace(' ', '')

    # Wrap the binary mask
    df['ptm_binary_mask'] = "Mask=<" + df['ptm_binary_mask'] + ">"

    # Shuffle the dataset
    # df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    return df

# Define instruction
instruction = "[Predict the ADP Ribosylation sites given the peptide sequence]"

# Load datasets
df_train = pd.read_csv('data/ADPr/adpr_sites_train.csv', dtype={'ptm_binary_mask': str})
df_test = pd.read_csv('data/ADPr/adpr_sites_test.csv', dtype={'ptm_binary_mask': str})

# Process datasets
df_train = prep_dataset(df_train, instruction)
df_test = prep_dataset(df_test, instruction)

print(f"Training set samples: {len(df_train)}")
print(f"Test set samples: {len(df_test)}")

Training set samples: 31176
Test set samples: 4246


Convert the tabular pandas dataframes into instruction datasets in the Alpaca format.

In [ ]:
# Convert the dataframes to JSON instruction dataset format
def df_to_json_instruction(df, input_col, output_col, instruction_col):
    dataset = []
    for _, row in df.iterrows():
        example = {
            "input": row[input_col],
            "output": row[output_col],
            "instruction": row[instruction_col]
        }
        dataset.append(example)
    return json.dumps(dataset, indent=4)

json_train = df_to_json_instruction(df_train, 'seq_sequence', 'ADPr_binding_sites', 'instruction')
json_test = df_to_json_instruction(df_test, 'seq_sequence', 'ADPr_binding_sites', 'instruction')

# Save JSON to file
train_path = 'data/ADPr/train.json'
test_path = 'data/ADPr/test.json'
with open(train_path, 'w') as f:
    f.write(json_train)
    print(f"Instruction dataset saved to {train_path}")
    f.close()
with open(test_path, 'w') as f:
    f.write(json_test)
    print(f"Instruction dataset saved to {test_path}")
    f.close()

Instruction dataset saved to data/ADPr/train.json
Instruction dataset saved to data/ADPr/test.json


Update the dataset_info.json so that the training set will appear in the LLaMA-Factory web UI.

In [ ]:
# Load the existing dataset_info.json
dataset_info_path = "/content/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    dataset_info = json.load(f)

# Create a new entry for the training set
dataset_info["adpr_train"] = {
    "file_name": "ADPr/train.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output",
        "history": None
    }
}

# Create a new entry for the test set
dataset_info["adpr_test"] = {
    "file_name": "ADPr/test.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output",
        "history": None
    }
}

# Write the updated dataset_info back to file
with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=4)

print("Entry 'adpr_training' has been added to dataset_info.json.")
print("Entry 'adpr_test' has been added to dataset_info.json.")

Entry 'adpr_training' has been added to dataset_info.json.
Entry 'adpr_test' has been added to dataset_info.json.


# Model Fine Tuning

Run this cell to open the web UI and configure the training procedure. You can run the procedure from the UI or export the equivalent CLI command and run it manually, as in the next step.

In [ ]:
!GRADIO_SHARE=1 llamafactory-cli webui

2025-04-09 01:30:38.756711: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-09 01:30:38.775339: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744162238.797099   76474 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744162238.803816   76474 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-09 01:30:38.826530: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

This runs the training command that was generated and exported from LLaMA-Factory.

In [ ]:
!llamafactory-cli train \
    --stage sft \
    --do_train True \
    --model_name_or_path GreatCaptainNemo/ProLLaMA_Stage_1 \
    --preprocessing_num_workers 16 \
    --finetuning_type lora \
    --template alpaca \
    --flash_attn auto \
    --dataset_dir data \
    --dataset adpr_train \
    --cutoff_len 2048 \
    --learning_rate 0.0003 \
    --num_train_epochs 8 \
    --max_samples 100000 \
    --per_device_train_batch_size 16 \
    --gradient_accumulation_steps 8 \
    --lr_scheduler_type cosine \
    --max_grad_norm 1.0 \
    --logging_steps 5 \
    --save_steps 500 \
    --warmup_steps 40 \
    --packing False \
    --report_to none \
    --output_dir saves/Custom/lora/train_20250716_1710 \
    --bf16 True \
    --plot_loss True \
    --trust_remote_code True \
    --ddp_timeout 180000000 \
    --include_num_input_tokens_seen True \
    --optim adamw_torch \
    --lora_rank 64 \
    --lora_alpha 128 \
    --lora_dropout 0.05 \
    --lora_target q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj \
    --val_size 0.1 \
    --eval_strategy steps \
    --eval_steps 100 \
    --per_device_eval_batch_size 16

Streaming output truncated to the last 5000 lines.
model-00001-of-00002.safetensors:  15% 1.47G/9.98G [00:30<01:46, 80.2MB/s]

model-00002-of-00002.safetensors:  68% 2.39G/3.50G [00:30<00:14, 78.9MB/s]
model-00001-of-00002.safetensors:  15% 1.48G/9.98G [00:30<01:45, 80.5MB/s]

model-00002-of-00002.safetensors:  69% 2.40G/3.50G [00:30<00:13, 78.9MB/s]
model-00001-of-00002.safetensors:  15% 1.49G/9.98G [00:30<01:45, 80.5MB/s]

model-00002-of-00002.safetensors:  69% 2.41G/3.50G [00:30<00:13, 79.3MB/s]
model-00001-of-00002.safetensors:  15% 1.50G/9.98G [00:30<01:44, 81.0MB/s]

model-00002-of-00002.safetensors:  69% 2.42G/3.50G [00:30<00:13, 79.4MB/s]
model-00001-of-00002.safetensors:  15% 1.51G/9.98G [00:30<01:44, 80.8MB/s]

model-00002-of-00002.safetensors:  69% 2.43G/3.50G [00:30<00:13, 79.5MB/s]
model-00001-of-00002.safetensors:  15% 1.52G/9.98G [00:30<01:44, 81.0MB/s]

model-00002-of-00002.safetensors:  70% 2.44G/3.50G [00:30<00:13, 79.2MB/s]
model-00001-of-00002.safetensors:  15% 1.53

Export the repository state to save a backup.

In [ ]:
!tar -czf /content/llama-factory-backup.tar.gz /content/LLaMA-Factory/
print("Archive created at /content/llama-factory-backup.tar.gz")

Load repository state from backup.

In [ ]:
!tar -xzf /content/drive/MyDrive/llama-factory-backup_2025-03-11-22-32.tar #-C /content/
print("Archive extracted.")

Archive extracted.


# Load Model
Load the final model.

In [ ]:
# Define paths
model_path = "/content/LLaMA-Factory/saves/Custom/lora/train_20250716_1710/checkpoint-1760-int8"

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

# Model Inference

Enter a prompt and generate a response.

In [ ]:
# Inference function
def generate_response(prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_length=max_length)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Define the query to match the training format
query = """
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
[Predict the ADP Ribosylation sites given the peptide sequence]
Seq=<PDLRASGGSGAGKAKKSVDKN>

### Response:
"""

# Run inference
response = generate_response(query, max_length=len(query) + 28)  # Adjust max_length for input + output length
print(response)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
[Predict the ADP Ribosylation sites given the peptide sequence]
Seq=<PDLRASGGSGAGKAKKSVDKN>

### Response:
 Sites=<S6,S9>


# Model Evaluation (Residue List)
Here we evaluate the performance of the model on the test set. Use this one if predicting a list of residues.

In [ ]:
# --------------------------------
#  Define subset of residues
# --------------------------------
amino_acids = ['D', 'E']
aa_set = set(amino_acids)

# --------------------------------
# Define paths
# --------------------------------
test_set_path = "/content/LLaMA-Factory/data/ADPr/test.json"
output_path = model_path + "/model_eval_results.csv"

# --------------------------------
#  Helpers
# --------------------------------
def generate_response(prompt, max_length=200):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(**inputs, max_length=max_length)
    return tokenizer.decode(output[0], skip_special_tokens=True)

def extract_sites(text):
    """Return list like ['R1','S4','E17'] from 'Sites=<R1,S4,E17>'."""
    marker = "Sites=<"
    start  = text.find(marker)
    if start == -1:
        return None
    start += len(marker)
    end = text.find(">", start)
    if end == -1:
        return None
    s = text[start:end].strip()
    return [] if s == "" else [x.strip() for x in s.split(",")]

import re
def extract_pos(site):                       # numeric position as int
    m = re.search(r"\d+", site)
    return int(m.group()) if m else None

def residue_letter(site):                    # first char assumed residue
    return site[0] if site else None

# --------------------------------
#  Main evaluation
# --------------------------------
with open(test_set_path) as f:
    test_set = json.load(f)

results = []
tp = fp = fn = 0                 # overall counts
tp_sub = fp_sub = fn_sub = 0     # subset counts
total_expected = total_expected_sub = 0
region_correct = 0

for i, sample in enumerate(test_set, 1):
    query = f"""
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{sample['instruction']}
{sample['input']}

### Response:
"""
    pred_text   = generate_response(query, max_length=len(query)+22)
    exp_sites   = extract_sites(sample["output"])
    pred_sites  = extract_sites(pred_text)

    results.append({"original sequence": sample["input"],
                    "actual sites": exp_sites,
                    "predicted sites": pred_sites})

    if exp_sites is None or pred_sites is None:      # skip bad parses
        continue

    # --- build position sets ---------------------------------------------
    exp_pos_set  = {extract_pos(s) for s in exp_sites  if extract_pos(s) is not None}
    pred_pos_set = {extract_pos(s) for s in pred_sites if extract_pos(s) is not None}

    # --- exact site metrics ----------------------------------------------
    tp += len(exp_pos_set & pred_pos_set)
    fp += len(pred_pos_set - exp_pos_set)
    fn += len(exp_pos_set - pred_pos_set)

    # --- region (±3) accuracy --------------------------------------------
    total_expected += len(exp_pos_set)
    for p in exp_pos_set:
        if any(abs(p - q) <= 3 for q in pred_pos_set):
            region_correct += 1

    # --- subset (selected amino acids) -----------------------------------
    exp_subset_pos = {extract_pos(s) for s in exp_sites
                      if residue_letter(s) in aa_set and extract_pos(s) is not None}
    pred_subset_pos = {extract_pos(s) for s in pred_sites
                       if residue_letter(s) in aa_set and extract_pos(s) is not None}

    total_expected_sub += len(exp_subset_pos)
    tp_sub += len(exp_subset_pos & pred_subset_pos)
    fp_sub += len(pred_subset_pos - exp_subset_pos)
    fn_sub += len(exp_subset_pos - pred_subset_pos)

    if i % 200 == 0:
        print(f"Processed {i}/{len(test_set)} samples...")

# --------------------------------
#  Aggregate metrics
# --------------------------------
def safe_div(num, den):     # avoid ZeroDivisionError
    return num / den if den else 0

# --- overall (all residues) ---------------------------------------------
precision     = safe_div(tp, tp + fp)
recall        = safe_div(tp, tp + fn)
f1            = safe_div(2*precision*recall, precision + recall)
site_acc      = safe_div(tp, total_expected)
region_acc    = safe_div(region_correct, total_expected)

# --- subset residues ---------------------------------------------------
precision_sub = safe_div(tp_sub, tp_sub + fp_sub)
recall_sub    = safe_div(tp_sub, tp_sub + fn_sub)
f1_sub        = safe_div(2*precision_sub*recall_sub, precision_sub + recall_sub)
site_acc_sub  = safe_div(tp_sub, total_expected_sub)

# --------------------------------
#  Save & report
# --------------------------------
pd.DataFrame(results).to_csv(output_path, index=False)

print("\n========  OVERALL METRICS (all amino acids)  ========")
print(f"Site-level Accuracy (exact):      {site_acc:.2%} ({tp}/{total_expected})")
print(f"Precision:                        {precision:.2%}")
print(f"Recall:                           {recall:.2%}")
print(f"F1-score:                         {f1:.2%}")
print(f"Region Accuracy (±3 positions):   {region_acc:.2%} ({region_correct}/{total_expected})")

print("\n\n========  SUBSET METRICS (only {aa})  ========".format(aa=','.join(amino_acids)))
print(f"Subset Site Accuracy (exact):     {site_acc_sub:.2%} ({tp_sub}/{total_expected_sub})")
print(f"Subset Precision:                 {precision_sub:.2%}")
print(f"Subset Recall:                    {recall_sub:.2%}")
print(f"Subset F1-score:                  {f1_sub:.2%}")

print(f"\nResults saved to {output_path}")

Processed 200/4246 samples...
Processed 400/4246 samples...
Processed 600/4246 samples...
Processed 800/4246 samples...
Processed 1000/4246 samples...
Processed 1200/4246 samples...
Processed 1400/4246 samples...
Processed 1600/4246 samples...
Processed 1800/4246 samples...
Processed 2000/4246 samples...
Processed 2200/4246 samples...
Processed 2400/4246 samples...
Processed 2600/4246 samples...
Processed 2800/4246 samples...
Processed 3000/4246 samples...
Processed 3200/4246 samples...
Processed 3400/4246 samples...
Processed 3600/4246 samples...
Processed 3800/4246 samples...
Processed 4000/4246 samples...
Processed 4200/4246 samples...

========  OVERALL METRICS (all amino acids)  ========
Site-level Accuracy (exact):      43.49% (2748/6318)
Precision:                        46.77%
Recall:                           43.49%
F1-score:                         45.08%
Region Accuracy (±3 positions):   63.90% (4037/6318)


========  SUBSET METRICS (only D,E)  ========
Subset Site Accuracy 

# Export Model to HuggingFace

Quantize the model and merge the LoRA adapter with the base model.

In [ ]:
# 1. load base + LoRA on GPU in fp16
model = AutoPeftModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).merge_and_unload()                       # merge here while still fp16

# 2. move to CPU + fp32, then INT-8-quantise
model = model.to(dtype=torch.float32, device="cpu")
model = quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)

# 3. save back to the same folder, filtering out non-tensor entries
state_dict = {k: v for k, v in model.state_dict().items() if isinstance(v, torch.Tensor)}
model.save_pretrained(model_path, state_dict=state_dict,
                      safe_serialization=True, max_shard_size="2GB")

AutoTokenizer.from_pretrained(model_path, use_fast=True).save_pretrained(model_path)

print("Model path file contents:")
!ls -lh {model_path}

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model path file contents:
total 1.1G
-rw-r--r-- 1 root root  867 Jul 16 23:36 adapter_config.json
-rw-r--r-- 1 root root 611M Jul 16 23:36 adapter_model.safetensors
-rw-r--r-- 1 root root  389 Jul 16 23:36 all_results.json
-rw-r--r-- 1 root root  638 Jul 17 01:28 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Jul 16 22:34 checkpoint-1000
drwxr-xr-x 2 root root 4.0K Jul 16 23:15 checkpoint-1500
drwxr-xr-x 2 root root 4.0K Jul 16 23:36 checkpoint-1760
drwxr-xr-x 2 root root 4.0K Jul 16 21:53 checkpoint-500
-rw-r--r-- 1 root root  728 Jul 17 01:28 config.json
-rw-r--r-- 1 root root  201 Jul 16 23:36 eval_results.json
-rw-r--r-- 1 root root  132 Jul 17 01:28 generation_config.json
-rw-r--r-- 1 root root 208K Jul 17 00:24 model_eval_results.csv
-rw-r--r-- 1 root root 502M Jul 17 01:28 model.safetensors
-rw-r--r-- 1 root root 1.5K Jul 16 23:36 README.md
-rw-r--r-- 1 root root  547 Jul 17 01:28 special_tokens_map.json
-rw-r--r-- 1 root root 1.1K Jul 17 01:28 tokenizer_config.json
-rw-r--r-- 

New dest folder version

In [ ]:
# ------------------------------------------------------------------
# source (read‑only) and destination (write) paths
# ------------------------------------------------------------------
src_path  = model_path               # existing base‑plus‑LoRA folder
dest_path = src_path + "-int8"       # or any other folder name
os.makedirs(dest_path, exist_ok=True)

# 1. load base + LoRA on GPU in fp16 and merge
model = AutoPeftModelForCausalLM.from_pretrained(
    src_path, device_map="auto", torch_dtype=torch.float16, low_cpu_mem_usage=True
).merge_and_unload()

# 2. move to CPU in fp32, then quantise to INT‑8
model = model.to(dtype=torch.float32, device="cpu")
model = quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)

# 3. save into the *destination* folder
state_dict = {k: v for k, v in model.state_dict().items() if isinstance(v, torch.Tensor)}
model.save_pretrained(dest_path, state_dict=state_dict,
                      safe_serialization=True, max_shard_size="2GB")

AutoTokenizer.from_pretrained(src_path, use_fast=True).save_pretrained(dest_path)

print("Destination folder contents:")
!ls -lh {dest_path}

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Destination folder contents:
total 506M
-rw-r--r-- 1 root root  638 Jul 17 04:35 chat_template.jinja
-rw-r--r-- 1 root root  728 Jul 17 04:35 config.json
-rw-r--r-- 1 root root  132 Jul 17 04:35 generation_config.json
-rw-r--r-- 1 root root 502M Jul 17 04:35 model.safetensors
-rw-r--r-- 1 root root  547 Jul 17 04:35 special_tokens_map.json
-rw-r--r-- 1 root root 1.1K Jul 17 04:35 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Jul 17 04:35 tokenizer.json
-rw-r--r-- 1 root root 489K Jul 17 04:35 tokenizer.model


Push the final model and LoRA adapter to the HuggingFace hub repository.

In [ ]:
# Get HuggingFace API token from Google Colab secret
hf_token = userdata.get('HF_API_TOKEN')

# Define repository details
username = "jbenbudd"
repo_name = "adpr-llama-int8"
repo_id = f"{username}/{repo_name}"

# Clone the repository
api = HfApi()
local_repo_dir = "local_model_repo"
repo = Repository(local_dir=local_repo_dir, clone_from=repo_id, use_auth_token=hf_token)

# List of files to include
files_to_copy = [
    "README.md",
    "model.safetensors",
    "model.safetensors.index.json",
    "all_results.json",
    "config.json",
    "chat_template.jinja",
    "eval_results.json",
    "generation_config.json",
    "llamaboard_config.yaml",
    "model_config.json",
    "model_eval_results.csv",
    "running_log.txt",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer.model",
    "tokenizer_config.json",
    "trainer_config.json",
    "trainer_log.jsonl",
    "trainer_state.json",
    "training_args.bin",
    "training_args.yaml",
    "training_eval_loss.png",
    "training_loss.png"
]

# Copy model files into the local repository folder
for filename in files_to_copy:
    src = os.path.join(model_path, filename)
    dst = os.path.join(local_repo_dir, filename)
    if os.path.exists(src):
        !cp "{src}" "{dst}"
    else:
        print(f"Skipping: {src} not found.")

# Commit and push changes to HuggingFace Hub
repo.git_add(auto_lfs_track=True)
repo.git_commit("train_1_epoch_test")
repo.git_push()

print("Model files have been pushed to Hugging Face Hub.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'Repository' (from 'huggingface_hub.repository') is deprecated and will be removed from version '1.0'. Please prefer the http-based alternatives instead. Given its large adoption in legacy code, the complete removal is only planned on next major release.
For more details, please read https://huggingface.co/docs/huggingface_hub/concepts/git_vs_http.
  warnings.warn(warning_message, FutureWarning)
Cloning https://huggingface.co/jbenbudd/adpr-llama-int8 into local empty directory.


Skipping: /content/LLaMA-Factory/saves/Custom/lora/train_20250716_1710/model.safetensors.index.json not found.
Skipping: /content/LLaMA-Factory/saves/Custom/lora/train_20250716_1710/llamaboard_config.yaml not found.
Skipping: /content/LLaMA-Factory/saves/Custom/lora/train_20250716_1710/model_config.json not found.
Skipping: /content/LLaMA-Factory/saves/Custom/lora/train_20250716_1710/running_log.txt not found.
Skipping: /content/LLaMA-Factory/saves/Custom/lora/train_20250716_1710/trainer_config.json not found.
Skipping: /content/LLaMA-Factory/saves/Custom/lora/train_20250716_1710/training_args.yaml not found.


Adding files tracked by Git LFS: ['training_eval_loss.png', 'training_loss.png']. This may take a bit of time if the files are large.


Upload file model.safetensors:   0%|          | 1.00/501M [00:00<?, ?B/s]

Upload file tokenizer.model:   0%|          | 1.00/488k [00:00<?, ?B/s]

Upload file training_args.bin:   0%|          | 1.00/5.55k [00:00<?, ?B/s]

Upload file training_eval_loss.png:   0%|          | 1.00/39.9k [00:00<?, ?B/s]

Upload file training_loss.png:   0%|          | 1.00/28.1k [00:00<?, ?B/s]

To https://huggingface.co/jbenbudd/adpr-llama-int8
   d3ac0da..ac37ed4  main -> main

   d3ac0da..ac37ed4  main -> main



Model files have been pushed to Hugging Face Hub.


Disconnect runtime

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
import time
while True:
    time.sleep(60)
    print("Alive")

Alive
Alive
Alive
Alive
Alive
Alive


KeyboardInterrupt: 